In [4]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [11]:
import torch
from src.config import DATASET_ROOT, POSE_DATASET_ROOT, GAMMA_POSE_DATASET_ROOT
from src.Skeleton_model.yolo_pose_tracking import save_annotated_pose_videos
from src.rwf2000 import RWF2000PoseDataset 
from src.Skeleton_model.graph import SkeletonGraph, compute_joint_distance_to_center_of_gravity
from src.Skeleton_model.stgcn import STGCN
from scripts.common.get_device import get_available_device
from ultralytics import YOLO
from pathlib import Path
import torch
import numpy as np
from src.Skeleton_model.yolo_pose_tracking import pose_data_to_stgcn_tensor

In [ ]:
pose_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train")
radii = compute_joint_distance_to_center_of_gravity(pose_dataset)
skeleton_graph = SkeletonGraph(radii)
device = get_available_device()
model = STGCN(adjacency=skeleton_graph.A).to(device)

In [6]:
from pathlib import Path
import torch
import numpy as np

def analyse_pose_database(pose_root):
    pose_files = list(Path(pose_root).rglob("*.pt"))

    empty_samples = 0
    detection_coverages = []
    tracking_coverages = []
    visible_joints_per_frame = []
    keypoint_confidences = []
    unique_track_ids_per_video = []

    for pose_path in pose_files:
        pose_data = torch.load(pose_path, weights_only=False)
        frames = pose_data["frames"]

        frames_with_detections = 0
        frames_with_tracks = 0
        total_visible_joints = 0
        track_ids = set()
        sample_has_pose = False

        for frame in frames:
            people = frame["people"]

            if len(people) > 0:
                frames_with_detections += 1
                sample_has_pose = True

            frame_has_track = False

            for person in people:
                if person["track_id"] is not None:
                    track_ids.add(person["track_id"])
                    frame_has_track = True

                confidence = person["keypoint_confidence"]
                visible = confidence > 0

                total_visible_joints += visible.sum().item()
                keypoint_confidences.extend(
                    confidence[visible].tolist()
                )

            if frame_has_track:
                frames_with_tracks += 1

        num_frames = len(frames)

        if not sample_has_pose:
            empty_samples += 1

        detection_coverages.append(
            frames_with_detections / num_frames
        )

        tracking_coverages.append(
            frames_with_tracks / num_frames
        )

        visible_joints_per_frame.append(
            total_visible_joints / num_frames
        )

        unique_track_ids_per_video.append(
            len(track_ids)
        )

    return {
        "num_samples": len(pose_files),
        "empty_samples": empty_samples,
        "mean_detection_coverage": np.mean(detection_coverages),
        "mean_tracking_coverage": np.mean(tracking_coverages),
        "mean_visible_joints_per_frame": np.mean(visible_joints_per_frame),
        "mean_keypoint_confidence": np.mean(keypoint_confidences),
        "mean_unique_track_ids_per_video": np.mean(unique_track_ids_per_video),
    }

In [7]:
original_results = analyse_pose_database(
    POSE_DATASET_ROOT
)

gamma_results = analyse_pose_database(
    GAMMA_POSE_DATASET_ROOT
)

print("ORIGINAL")
for key, value in original_results.items():
    print(f"{key}: {value}")

print("\nGAMMA")
for key, value in gamma_results.items():
    print(f"{key}: {value}")

ORIGINAL
num_samples: 2000
empty_samples: 288
mean_detection_coverage: 0.6915333333333334
mean_tracking_coverage: 0.6915333333333334
mean_visible_joints_per_frame: 28.619839999999996
mean_keypoint_confidence: 0.6205726204782711
mean_unique_track_ids_per_video: 6.101

GAMMA
num_samples: 2000
empty_samples: 299
mean_detection_coverage: 0.6793166666666666
mean_tracking_coverage: 0.6793166666666666
mean_visible_joints_per_frame: 27.467806666666668
mean_keypoint_confidence: 0.6285023874831771
mean_unique_track_ids_per_video: 5.991


In [8]:
from pathlib import Path
import torch


def pose_quality_score(pose_data):
    frames_with_tracks = 0
    visible_joints = 0
    confidences = []

    for frame in pose_data["frames"]:
        if len(frame["people"]) > 0:
            frames_with_tracks += 1

        for person in frame["people"]:
            confidence = person["keypoint_confidence"]
            visible = confidence > 0

            visible_joints += visible.sum().item()
            confidences.extend(confidence[visible].tolist())

    mean_confidence = (
        sum(confidences) / len(confidences)
        if confidences else 0.0
    )

    return frames_with_tracks, visible_joints, mean_confidence


def compare_pose_databases(original_root, gamma_root):
    original_root = Path(original_root)
    gamma_root = Path(gamma_root)

    original_wins = 0
    gamma_wins = 0
    ties = 0

    for original_path in original_root.rglob("*.pt"):
        relative_path = original_path.relative_to(original_root)
        gamma_path = gamma_root / relative_path

        original_pose = torch.load(original_path, weights_only=False)
        gamma_pose = torch.load(gamma_path, weights_only=False)

        original_score = pose_quality_score(original_pose)
        gamma_score = pose_quality_score(gamma_pose)

        if gamma_score > original_score:
            gamma_wins += 1
        elif original_score > gamma_score:
            original_wins += 1
        else:
            ties += 1

    print("Original selected:", original_wins)
    print("Gamma selected:   ", gamma_wins)
    print("Ties:             ", ties)

In [9]:
compare_pose_databases(
    POSE_DATASET_ROOT,
    GAMMA_POSE_DATASET_ROOT,
)

Original selected: 1150
Gamma selected:    565
Ties:              285


In [15]:
def score(tensor):
    conf = tensor[2]
    visible = conf > 0

    frames_with_pose = visible.any(dim=1).sum().item()
    visible_joints = visible.sum().item()

    return frames_with_pose, visible_joints


original = gamma = ties = 0

for original_path in Path(POSE_DATASET_ROOT).rglob("*.pt"):

    relative = original_path.relative_to(POSE_DATASET_ROOT)
    gamma_path = Path(GAMMA_POSE_DATASET_ROOT) / relative

    original_pose = torch.load(original_path, weights_only=False)
    gamma_pose = torch.load(gamma_path, weights_only=False)

    original_score = score(
        pose_data_to_stgcn_tensor(original_pose, max_people=4)
    )

    gamma_score = score(
        pose_data_to_stgcn_tensor(gamma_pose, max_people=4)
    )

    if original_score > gamma_score:
        original += 1
    elif gamma_score > original_score:
        gamma += 1
    else:
        ties += 1

print(f"Original selected: {original}")
print(f"Gamma selected:    {gamma}")
print(f"Ties:              {ties}")

Original selected: 1039
Gamma selected:    514
Ties:              447
